# Этап 6 - FT-Transformer

В этом ноутбуке обучается FT-Transformer для бинарной классификации риска повторной госпитализации. По структуре этап похож на MLP: используются те же подготовленные `train`, `val` и `test` split, отдельно обрабатываются числовые и категориальные признаки, а финальный порог выбирается по F2 на validation.

Главное отличие от MLP в том, что здесь табличные признаки обрабатывает специализированная Transformer-архитектура из `rtdl_revisiting_models`. Она строит представления признаков и применяет attention-блоки, поэтому может учитывать взаимодействия между признаками иначе, чем обычная полносвязная сеть.


## 1. Настройка окружения

In [ ]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

import rtdl_revisiting_models as rtdl

# Добавляем папку src, чтобы импортировать общие функции метрик проекта.
sys.path.append("../src")
import utils

# Основные настройки эксперимента и пути к данным/артефактам.
SEED = 42
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
METRICS_DIR = RESULTS_DIR / "metrics"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"

# Папки создаются заранее, чтобы сохранение результатов не упало в конце ноутбука.
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

## 2. Загрузка подготовленных данных

На этом шаге загружаются уже подготовленные `train`, `val` и `test` split из `data/processed`.

Отдельно читается `feature_types.json`, где сохранены списки числовых и категориальных признаков. Это пересекается с MLP-ноутбуком: обе модели используют одинаковые входные данные и одинаковое разделение признаков. Так сравнение получается честнее, потому что модели отличаются архитектурой, а не набором данных.


In [ ]:
# Загружаем готовые split, полученные на этапе preprocessing.
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

# feature_types хранит разделение признаков на числовые и категориальные.
with open(PROCESSED_DIR / "feature_types.json", encoding="utf-8") as f:
    feature_types = json.load(f)

numeric_features = feature_types["numeric"]
categorical_features = feature_types["categorical"]
target_col = "target"

# Для похожести с MLP-ноутбуком оставляем короткие алиасы.
num_cols = numeric_features
cat_cols = categorical_features

# Быстрая проверка размеров и дисбаланса классов.
print(f"train: {train.shape}  val: {val.shape}  test: {test.shape}")
print(f"Числовые: {len(num_cols)}  Категориальные: {len(cat_cols)}")
print(f"Доля положительного класса в train: {train[target_col].mean():.4f}")

## 3. Выделение целевой переменной

На этом шаге из каждого split отдельно выделяется столбец `target`. Он переводится в `np.float32`, потому что дальше используется бинарная функция потерь `BCEWithLogitsLoss`, которая ожидает вещественные целевые значения.

Это такой же смысловой шаг, как в MLP: признаки подаются в модель, а `target` используется только для расчёта loss и метрик.


In [ ]:
# Целевая переменная нужна отдельно от признаков.
# float32 подходит для BCEWithLogitsLoss в бинарной классификации.
y_train = train[target_col].values.astype(np.float32)
y_val = val[target_col].values.astype(np.float32)
y_test = test[target_col].values.astype(np.float32)

## 4. Кодирование категорий

Категориальные признаки нельзя подавать в модель как строки. Поэтому для каждого категориального столбца строится словарь `значение -> индекс`, причём словарь обучается только на train.

В MLP эти индексы дальше идут в `nn.Embedding`. В FT-Transformer идея похожая: категориальные признаки тоже передаются как индексы, а сама библиотека строит для них внутренние представления. Индекс `0` зарезервирован для неизвестных категорий, которые могут встретиться в validation или test.


In [ ]:
# Маппинг строка -> индекс строится только по train, чтобы не было утечки из val/test.
# Индекс 0 зарезервирован для неизвестных категорий в validation/test.
cat_vocabs: dict[str, dict] = {}
for c in cat_cols:
    vals = train[c].astype("string").fillna("Unknown").unique().tolist()
    cat_vocabs[c] = {v: i + 1 for i, v in enumerate(vals)}

# Размер словаря + 1, потому что индекс 0 занят под неизвестные значения.
cat_cardinalities = [len(cat_vocabs[c]) + 1 for c in cat_cols]


def encode_cats(df: pd.DataFrame, vocabs: dict, cols: list[str]) -> np.ndarray:
    """Заменяет категориальные значения их индексами для embedding-представлений."""
    out = np.zeros((len(df), len(cols)), dtype=np.int64)
    for j, c in enumerate(cols):
        vocab = vocabs[c]
        values = df[c].astype("string").fillna("Unknown")
        # Если категории не было в train, ставим 0.
        out[:, j] = [vocab.get(v, 0) for v in values]
    return out


# Получаем матрицы индексов категорий для каждого split.
X_train_cat = encode_cats(train, cat_vocabs, cat_cols)
X_val_cat = encode_cats(val, cat_vocabs, cat_cols)
X_test_cat = encode_cats(test, cat_vocabs, cat_cols)

print("Категориальные признаки закодированы - train:", X_train_cat.shape, " val:", X_val_cat.shape, " test:", X_test_cat.shape)
print("Максимальный размер словаря:", max(cat_cardinalities), "всего категорий:", sum(cat_cardinalities))

## 5. Нормализация числовых признаков

Числовые признаки переводятся к сопоставимому масштабу через `StandardScaler`. Если признаки имеют разные диапазоны, оптимизатору сложнее стабильно обновлять веса.

Scaler обучается только на train, а затем применяется к validation и test. Это полностью совпадает с логикой MLP-ноутбука и защищает от утечки данных из проверочных выборок.


In [ ]:
# StandardScaler обучается только на train.
scaler = StandardScaler()

# Числовые признаки приводятся к масштабу со средним 0 и стандартным отклонением 1.
X_train_num = scaler.fit_transform(train[num_cols].values).astype(np.float32)

# Для val/test используем параметры scaler из train, чтобы не было утечки данных.
X_val_num = scaler.transform(val[num_cols].values).astype(np.float32)
X_test_num = scaler.transform(test[num_cols].values).astype(np.float32)

print("Числовые признаки нормализованы - train:", X_train_num.shape, " val:", X_val_num.shape, " test:", X_test_num.shape)

## 6. Подготовка тензоров и DataLoader

После кодирования и нормализации данные переводятся в PyTorch-тензоры нужных типов. Категориальные признаки становятся `long`, потому что это индексы категорий, а числовые признаки и целевая переменная становятся `float32`.

Для оценки используются отдельные DataLoader для train, validation и test. Во время обучения train-loader создаётся внутри функции `train_candidate`, потому что batch size является гиперпараметром и может отличаться у разных конфигураций.


In [ ]:
def make_dataset(X_num: np.ndarray, X_cat: np.ndarray, y: np.ndarray) -> TensorDataset:
    """Переводит numpy-массивы в TensorDataset для PyTorch."""
    return TensorDataset(
        # Числовые признаки участвуют в вычислениях модели как float32.
        torch.tensor(X_num, dtype=torch.float32),
        # Категории идут как индексы, поэтому нужен long.
        torch.tensor(X_cat, dtype=torch.long),
        # Target нужен для BCEWithLogitsLoss, поэтому float32.
        torch.tensor(y, dtype=torch.float32),
    )


train_dataset = make_dataset(X_train_num, X_train_cat, y_train)
val_dataset = make_dataset(X_val_num, X_val_cat, y_val)
test_dataset = make_dataset(X_test_num, X_test_cat, y_test)


def make_loader(dataset: TensorDataset, batch_size: int, shuffle: bool = False) -> DataLoader:
    # Фиксируем generator, чтобы перемешивание train было воспроизводимым.
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


# Для evaluation можно брать батчи крупнее, потому что backward pass не выполняется.
eval_batch_size = 2048
train_eval_loader = make_loader(train_dataset, eval_batch_size)
val_loader = make_loader(val_dataset, eval_batch_size)
test_loader = make_loader(test_dataset, eval_batch_size)

print(f"DataLoader для оценки готов: batch_size={eval_batch_size}")

## 7. Архитектура FT-Transformer

В MLP категориальные признаки проходят через embedding-слои, затем объединяются с числовыми признаками и идут в полносвязную сеть. FT-Transformer тоже работает с числовыми и категориальными признаками отдельно, но дальше использует Transformer-блоки.

Внутри `rtdl.FTTransformer` категориальные признаки превращаются в embedding-представления, числовые признаки тоже приводятся к внутреннему представлению, после чего attention-блоки ищут взаимодействия между признаками. На выходе модель возвращает один logit для бинарной классификации. `sigmoid` внутри модели не вызывается, потому что `BCEWithLogitsLoss` работает напрямую с logits.


## 8. Обучение с подбором порога

Здесь задаётся процедура обучения одной конфигурации FT-Transformer. Как и в MLP, из-за дисбаланса классов используется `BCEWithLogitsLoss` с `pos_weight = N_neg / N_pos`. Это увеличивает штраф за ошибки на редком положительном классе.

После каждой эпохи модель оценивается на validation. Бинарные предсказания строятся не по фиксированному порогу `0.5`, а по порогу, который максимизирует F2 на validation. F2 выбран потому, что он сильнее учитывает recall, а в задаче реадмиссии пропуск пациента из группы риска обычно хуже лишнего ложного срабатывания.


In [ ]:
@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    """Возвращает истинные метки и вероятности положительного класса."""
    model.eval()
    probs = []
    targets = []
    for x_num, x_cat, y in loader:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)
        logits = model(x_num, x_cat).squeeze(1)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        targets.append(y.numpy())
    return np.concatenate(targets).astype(int), np.concatenate(probs)


def train_candidate(params: dict) -> dict:
    # Каждая конфигурация стартует с одного seed, чтобы сравнение было стабильнее.
    set_seed(SEED)
    batch_size = params["batch_size"]
    train_loader = make_loader(train_dataset, batch_size=batch_size, shuffle=True)

    # FT-Transformer отдельно обрабатывает числовые и категориальные признаки.
    model = rtdl.FTTransformer(
        n_cont_features=len(num_cols),
        cat_cardinalities=cat_cardinalities,
        d_out=1,
        n_blocks=params["n_blocks"],
        d_block=params["d_block"],
        attention_n_heads=params["attention_n_heads"],
        attention_dropout=params["attention_dropout"],
        ffn_d_hidden=None,
        ffn_d_hidden_multiplier=4 / 3,
        ffn_dropout=params["ffn_dropout"],
        residual_dropout=params["residual_dropout"],
    ).to(device)

    # Вес положительного класса компенсирует сильный дисбаланс target.
    pos_weight_value = (len(y_train) - y_train.sum()) / y_train.sum()
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=device))
    optimizer = torch.optim.AdamW(
        model.make_parameter_groups(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    history = []
    best_state = None
    best_epoch = 0
    best_score = (-1.0, -1.0)
    best_threshold = 0.5
    bad_epochs = 0

    for epoch in tqdm(range(1, params["max_epochs"] + 1), desc=params["name"]):
        model.train()
        total_loss = 0.0
        for x_num, x_cat, y in train_loader:
            x_num = x_num.to(device)
            x_cat = x_cat.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x_num, x_cat).squeeze(1)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu()) * len(y)

        # На validation подбираем лучший порог именно для F2.
        _, val_proba = predict_proba(model, val_loader)
        threshold = utils.find_best_threshold_f2(y_val.astype(int), val_proba)
        val_metrics = utils.compute_metrics(y_val.astype(int), val_proba, threshold)
        row = {
            "epoch": epoch,
            "train_loss": total_loss / len(train_dataset),
            "threshold": threshold,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        # Главный критерий выбора - val_f2, а ROC-AUC используется как дополнительный критерий.
        score = (val_metrics["f2"], val_metrics["roc_auc"])
        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_threshold = threshold
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= params["patience"]:
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    model.to(device)
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
        "best_score": best_score,
    }


print("Функции обучения и предсказания готовы")

## 9. Подбор гиперпараметров на validation

В этом разделе сравниваются две конфигурации FT-Transformer: компактная `ft_small` и более крупная `ft_medium`. Для каждой конфигурации запускается одинаковая процедура обучения, после чего сохраняются лучшая эпоха, подобранный порог и validation-метрики.

Таблица сортируется по `val_f2`, потому что именно F2 лучше соответствует цели проекта: находить как можно больше положительных случаев, не игнорируя precision полностью. Test здесь не используется, как и в MLP-ноутбуке.


In [ ]:
# Небольшая сетка гиперпараметров: сравниваем компактную и более крупную модель.
param_grid = [
    {
        "name": "ft_small",
        "n_blocks": 1,
        "d_block": 32,
        "attention_n_heads": 4,
        "attention_dropout": 0.10,
        "ffn_dropout": 0.10,
        "residual_dropout": 0.00,
        "lr": 3e-4,
        "weight_decay": 1e-5,
        "batch_size": 1024,
        "max_epochs": 18,
        "patience": 5,
    },
    {
        "name": "ft_medium",
        "n_blocks": 2,
        "d_block": 64,
        "attention_n_heads": 4,
        "attention_dropout": 0.15,
        "ffn_dropout": 0.10,
        "residual_dropout": 0.00,
        "lr": 2e-4,
        "weight_decay": 1e-5,
        "batch_size": 1024,
        "max_epochs": 18,
        "patience": 5,
    },
]

search_rows = []
trained = []

for params in param_grid:
    result = train_candidate(params)
    last_best = result["history"][result["best_epoch"] - 1]
    row = {
        "name": params["name"],
        "best_epoch": result["best_epoch"],
        "threshold": result["best_threshold"],
        **{k: v for k, v in params.items() if k not in {"name"}},
        **{k: v for k, v in last_best.items() if k.startswith("val_")},
    }
    search_rows.append(row)
    trained.append({"params": params, **result})

# Лучшая строка будет первой: сначала максимизируем val_f2, затем val_roc_auc.
search_df = pd.DataFrame(search_rows).sort_values(["val_f2", "val_roc_auc"], ascending=False)
search_df

## 10. Предсказания лучшей модели

После подбора гиперпараметров берётся первая строка `search_df`, то есть лучшая модель по validation F2. Восстанавливается именно лучшая сохранённая версия весов, а не просто последняя эпоха.

Затем считаются вероятности положительного класса для train, validation и test. Эти вероятности нужны и для расчёта метрик, и для сохранения CSV-файлов с предсказаниями.


In [ ]:
# Берём лучшую модель из таблицы подбора гиперпараметров.
best_idx = search_df.index[0]
best_result = trained[best_idx]
best_model = best_result["model"]
best_threshold = best_result["best_threshold"]

# Считаем вероятности положительного класса для всех трёх выборок.
train_true, train_proba = predict_proba(best_model, train_eval_loader)
val_true, val_proba = predict_proba(best_model, val_loader)
test_true, test_proba = predict_proba(best_model, test_loader)

print("Лучшая конфигурация:", best_result["params"]["name"])
print("Лучший порог классификации:", round(best_threshold, 4))
print("Лучшая эпоха обучения:", best_result["best_epoch"])

## 11. Итоговые метрики

Финальный порог выбирается только на validation split по максимуму F2. После выбора он фиксируется и без дополнительной подстройки применяется к train, validation и test.



In [ ]:
# Собираем метрики, параметры и историю обучения в один словарь для сохранения.
metrics = {
    "model": "transformer",
    "architecture": "FT-Transformer from rtdl_revisiting_models",
    "selection_metric": "validation F2 with validation-optimized threshold",
    "best_params": best_result["params"],
    "best_epoch": best_result["best_epoch"],
    "threshold": best_threshold,
    "train": utils.compute_metrics(train_true, train_proba, best_threshold),
    "val": utils.compute_metrics(val_true, val_proba, best_threshold),
    "test": utils.compute_metrics(test_true, test_proba, best_threshold),
    "hyperparameter_search": search_rows,
    "training_history": [
        {"name": item["params"]["name"], "history": item["history"]}
        for item in trained
    ],
    "preprocessing": {
        "numeric_features": num_cols,
        "categorical_features": cat_cols,
        "cat_cardinalities": cat_cardinalities,
    },
}

metrics_df = pd.DataFrame([metrics[split] for split in ["train", "val", "test"]], index=["train", "val", "test"])
metrics_df


## 12. Сохранение артефактов

После оценки сохраняется JSON с метриками, выбранным порогом, гиперпараметрами и историей обучения. Такой файл нужен, чтобы сравнительные ноутбуки могли читать результаты автоматически, без ручного переноса чисел.

Также сохраняются CSV с вероятностями для validation и test. Они позволяют позже строить графики, сравнивать модели по одним и тем же объектам или анализировать пороги без повторного обучения FT-Transformer.


In [ ]:
def save_predictions(path: Path, y_true: np.ndarray, y_proba: np.ndarray, threshold: float) -> None:
    # Сохраняем не только класс, но и вероятность: она пригодится для графиков и сравнения порогов.
    pd.DataFrame(
        {
            "y_true": y_true.astype(int),
            "y_proba": y_proba,
            "y_pred": (y_proba >= threshold).astype(int),
        }
    ).to_csv(path, index=False)


save_predictions(PREDICTIONS_DIR / "transformer_val.csv", val_true, val_proba, best_threshold)
save_predictions(PREDICTIONS_DIR / "transformer_test.csv", test_true, test_proba, best_threshold)

with open(METRICS_DIR / "transformer.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Сохранён файл метрик:", METRICS_DIR / "transformer.json")
print("Сохранены предсказания validation:", PREDICTIONS_DIR / "transformer_val.csv")
print("Сохранены предсказания test:", PREDICTIONS_DIR / "transformer_test.csv")


## 13. Вывод

FT-Transformer обучен по тому же общему протоколу, что и MLP: preprocessing обучался только на train, гиперпараметры и порог выбирались на validation, а test использовался только для финальной оценки. За счёт этого результаты можно честно сравнивать с другими моделями проекта.

Метрики на test:

| Метрика | Значение |
|---|---:|
| ROC-AUC | 0.6580 |
| Precision | 0.1083 |
| Recall | 0.8359 |
| F2 | 0.3567 |

Лучшей конфигурацией стала `ft_medium`: 2 Transformer-блока, размер блока 64, 4 attention-heads, learning rate `0.0002`. Лучший порог классификации равен `0.3605`, он был выбран на validation по максимуму F2. Лучшая эпоха - 9, после неё качество по выбранному критерию уже не улучшалось достаточно стабильно.

Главный результат модели - высокий recall. На test она нашла 1049 положительных случаев и пропустила 206, то есть обнаружила примерно 83.6% пациентов из положительного класса. Это соответствует цели задачи: лучше чаще выделять пациентов с риском повторной госпитализации, чем пропускать их.

Минус модели - низкий precision: среди объектов, которые модель относит к положительному классу, действительно положительных немного. На test получилось 8634 false positive, поэтому модель даёт много лишних срабатываний. Это ожидаемо для несбалансированной медицинской задачи, особенно при оптимизации под F2, где recall важнее precision.

Метрики train, validation и test находятся примерно на одном уровне: ROC-AUC около 0.65-0.67, F2 около 0.36. Это значит, что сильного переобучения не видно: модель не показывает резко лучшее качество на train по сравнению с test. В сравнении с MLP FT-Transformer ведёт себя похоже по общей логике: высокий recall, низкий precision и умеренный ROC-AUC. Сохранённые артефакты совместимы с `08_comparison.ipynb`, поэтому результат можно использовать в общей таблице сравнения.
